# Week 8 (Notebook 1): Decoder‑Only Generation (GPT)

This notebook has two goals:

1. **Architecture intuition (PyTorch):** implement a tiny decoder‑only Transformer (causal self‑attention) to understand the forward pass + masking + next‑token loss.
2. **Power of pretraining (Hugging Face):** fine‑tune a small pretrained GPT model on a small dataset slice with **CPU‑friendly defaults**.

Notes:
- First run will download a dataset/model from Hugging Face.
- CPU training is intentionally small (few steps, short sequence length) so it finishes in a reasonable time.

In [ ]:
# Setup
import os
import math
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

seed = 204
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device

## Part A — Tiny decoder‑only Transformer in pure PyTorch

This is **not** meant to be competitive. It exists to make the GPT mechanics concrete:
- token embeddings + positional embeddings
- **causal mask** (prevent attending to future tokens)
- stacked Transformer blocks
- LM head projecting hidden states → vocab logits
- next‑token prediction loss

In [ ]:
@dataclass
class TinyGPTConfig:
    vocab_size: int
    block_size: int = 64
    d_model: int = 128
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.1


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: TinyGPTConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.cfg = cfg
        self.head_dim = cfg.d_model // cfg.n_heads
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)

        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("causal_mask", mask.view(1, 1, cfg.block_size, cfg.block_size))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.cfg.n_heads, self.head_dim).transpose(1, 2)  # (B, nh, T, hd)
        k = k.view(B, T, self.cfg.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.cfg.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, nh, T, T)
        att = att.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v  # (B, nh, T, hd)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, cfg: TinyGPTConfig):
        super().__init__()
        self.fc = nn.Linear(cfg.d_model, 4 * cfg.d_model)
        self.proj = nn.Linear(4 * cfg.d_model, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        x = self.fc(x)
        x = F.gelu(x)
        x = self.proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):
    def __init__(self, cfg: TinyGPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, cfg: TinyGPTConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size
        pos = torch.arange(0, T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


def tiny_generate(model, idx, max_new_tokens=50, temperature=1.0):
    model.eval()
    idx = idx.clone()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.cfg.block_size :]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / max(temperature, 1e-6)
        probs = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)
    return idx

In [ ]:
# Tiny demo dataset (character-level) so everything is self-contained.
text = """
to be or not to be.
this is a tiny corpus for a tiny gpt.
we only want to demonstrate causal masking and next-token loss.
""".strip().lower()

chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
vocab_size, len(data)

In [ ]:
# CPU-friendly tiny training loop (few steps)
cfg = TinyGPTConfig(vocab_size=vocab_size, block_size=64, d_model=128, n_heads=4, n_layers=2, dropout=0.1)
model = TinyGPT(cfg).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

def get_batch(batch_size=16):
    ix = torch.randint(0, len(data) - cfg.block_size - 1, (batch_size,))
    x = torch.stack([data[i : i + cfg.block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + cfg.block_size + 1] for i in ix])
    return x.to(device), y.to(device)

for step in range(200):
    x, y = get_batch(batch_size=16)
    _, loss = model(x, targets=y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if (step + 1) % 50 == 0:
        print(f"step {step+1:4d} | loss {loss.item():.4f}")

prompt = "to be"
idx0 = torch.tensor([[stoi[c] for c in prompt.lower()]], dtype=torch.long).to(device)
out = tiny_generate(model, idx0, max_new_tokens=80, temperature=0.9)[0].tolist()
print("".join(itos[i] for i in out))

## Part B — Fine‑tuning a pretrained GPT model (Hugging Face)

We now fine‑tune a pretrained decoder‑only model to show how much better it is with transfer learning.

**CPU‑realistic defaults**
- use `distilgpt2`
- short `block_size`
- small dataset slice
- `max_steps` instead of full epochs
- small batch size + gradient accumulation

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

hf_model_name = "distilgpt2"
block_size = 128

# Keep this small for CPU
n_train = 4000
n_val = 500

raw = load_dataset("wikitext", "wikitext-2-v1")
raw

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(examples["text"], return_attention_mask=False)

tok = raw.map(tokenize_fn, batched=True, remove_columns=raw["train"].column_names)

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = (len(concatenated["input_ids"]) // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_ds = tok.map(group_texts, batched=True)

# Shuffle then take small CPU-friendly slices
train_ds = lm_ds["train"].shuffle(seed=seed).select(range(min(n_train, len(lm_ds["train"])) ))
val_ds = lm_ds["validation"].shuffle(seed=seed).select(range(min(n_val, len(lm_ds["validation"])) ))
train_ds, val_ds

In [ ]:
model = AutoModelForCausalLM.from_pretrained(hf_model_name)
model.resize_token_embeddings(len(tokenizer))
model.to(device)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

out_dir = "./models/week8_distilgpt2_wikitext2"
os.makedirs(out_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=out_dir,
    overwrite_output_dir=True,
    max_steps=300,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=30,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    seed=seed,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)

trainer

In [ ]:
# CPU-realistic run: keep this small
# trainer.train()

# If you already trained, point to a checkpoint folder here:
# ckpt = "./models/week8_distilgpt2_wikitext2/checkpoint-300"
# model = AutoModelForCausalLM.from_pretrained(ckpt).to(device)

print("Ready: uncomment trainer.train() to fine-tune.")

In [ ]:
# Generation demo (works before/after fine-tuning)
from transformers import set_seed

set_seed(seed)
prompt = "The meaning of life is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

gen = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    top_p=0.95,
    temperature=0.9,
)
print(tokenizer.decode(gen[0], skip_special_tokens=True))